# DWBI Week 2 — DW Lifecycle & Architecture
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

Case: **PakFreight** — three transport modes, three files, three totals, zero trust.
The week ends with one trusted table: `Shipments`, 34 rows, 578,650 kg.

## Task 5 — Latency, drift, lineage and the people

### 1. Data latency

| | Consumer | Freshness needed | Why |
|---|---|---|---|
| a | Dispatcher rerouting around a blocked motorway | **Real-time** | A decision in minutes on live positions; yesterday's data is useless |
| b | CFO's monthly cost-per-tonne report | **Daily batch** | Monthly grain — intraday freshness adds nothing |
| c | Customer tracking a consignment online | **Near-real-time** | "Left today" needs hours-old data, not yesterday's |
| d | Board's quarterly mode-profitability review | **Weekly / batch** | Quarterly grain; stability beats speed |

Why "always real-time" is wrong — two concrete costs: **(1)** infrastructure cost: streaming pipelines,
ODS upkeep and monitoring for data nobody needs fresh (the board will not read it faster); **(2)**
correctness cost: real-time skips the staging quality checkpoint — the TOTAL row and the tons/kg trap
would flow straight into the CFO's report before anyone validated them.

![Latency decision flowchart](charts/t5_latency.png)

### 2. Schema drift

| Change | What happens on next refresh | How to detect it before a report |
|---|---|---|
| Rail adds `Wagon_Type` | Pipeline **carries on** — the new column is ignored (not in the contract), so its data is **silently dropped** | A staging check comparing source column list vs contract; alert on any unmapped column |
| `Freight Charges (PKR)` → `Charges_PKR` | Pipeline **breaks loudly** — the rename step references a column that no longer exists | Same column-list check, plus a Refresh smoke test in dev: row count and 578,650 must still hold |

![Schema drift flowchart](charts/t5_drift.png)

### 3. Lineage of 578,650 kg

![Lineage of 578,650 kg](charts/t5_lineage.png)

The CFO says the number looks too low. Using only the lineage, check in order:
**① the staging filters** — was a real row dropped (e.g. an over-aggressive duplicate rule?);
**② the rail conversion** — is `tons × 1000` still applied (a drifted header would silently shrink rail to 295 kg);
**③ the TOTAL-row guard** — did a summary row leak back in (which would raise, not lower, the total — ruling it out fast).

### 4. Roles in the BI ecosystem

| | Responsibility | Owner | Main tier |
|---|---|---|---|
| a | Designing the star schema for Shipments | **Data architect** | Storage / Presentation |
| b | Writing and maintaining the Power Query pipeline | **ETL / data engineer** | Staging |
| c | Agreeing what "delivered" officially means | **Business / data steward** | Staging (rules) |
| d | Building the dashboard Ops opens each morning | **BI developer** | Presentation |
| e | Keeping the rail export in its agreed format | **Source system owner** | Sources |

### 5. Responsibility map

**Staging** is owned by the ETL engineer, working to rules the steward agrees; **storage** by the data
architect; **presentation** by the BI developer — with the source owner contracted on format.
When nobody owns a tier, the stovepipe returns: Task 3's three totals happened precisely because no one
owned staging — each department owned *its* file end-to-end, so three private rule-sets produced three
"true" numbers and nobody could say which was right.

### 6. Character — discipline

A freight analyst who skips validation is a risk because every downstream decision — pricing, capacity,
the CFO's cost-per-tonne — multiplies whatever error they publish. If 578,650 were wrong and nobody
checked, PakFreight could under-price February freight on a phantom 83 tonnes of TOTAL-row weight,
promising capacity it never moved. Validation is not bureaucracy; it is the moment the analyst signs
their name under a number the business will spend money on.

### The question to answer

Handed an unreproducible dashboard, investigate in this order: **lineage first** — can I trace the number
to file → row → transformation, or does the chain break (missing staging, hand-edits)? **Then schema
drift** — did a renamed/added column silently change what the query reads since the dashboard was built?
**Then latency** — am I comparing a real-time figure against a batch figure, i.e. is the "difference"
just a different refresh moment? Lineage tells you *what* the number is; drift tells you *what changed*;
latency tells you *when* it was true.

In [1]:
# Latency decision helper: freshness should match the decision's clock, never faster
decisions = {
    "dispatcher reroute": ("real-time", "minutes decide it"),
    "CFO monthly report": ("daily batch", "monthly grain"),
    "customer tracking":  ("near-real-time", "hours-old is fine"),
    "board quarterly":    ("weekly/batch", "quarterly grain"),
}
for k, (fresh, why) in decisions.items():
    print(f"{k:22s} -> {fresh:15s} ({why})")
print("\nRule: the cheapest freshness that still beats the decision clock wins.")

dispatcher reroute     -> real-time       (minutes decide it)
CFO monthly report     -> daily batch     (monthly grain)
customer tracking      -> near-real-time  (hours-old is fine)
board quarterly        -> weekly/batch    (quarterly grain)

Rule: the cheapest freshness that still beats the decision clock wins.
